# 🌸 Flower Distributed Federated Learning Pipeline via gRPC (FL-IoT-IDS)
## Decentralized Edge Intrusion Detection over TCP/IP gRPC Sockets
### Graduation Thesis: *Xây dựng và đánh giá prototype hệ thống phát hiện xâm nhập IoT sử dụng Federated Learning trong môi trường dữ liệu non-IID*
**Author:** Nguyễn Việt Kỳ Quân (23521267) | **Advisor:** PGS.TS Lê Trung Quân | **Institution:** VNU-HCM UIT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/flower_federated_pipeline.ipynb)

**Repository:** `NVKQ2022/FedLearning` | **Transport:** Decoupled Edge Multi-Process via gRPC (TCP/IP Port 8080)


---
### 🛠️ Step 1: Environment Setup & gRPC Transport Verification
We verify GPU acceleration, install `flwr`, synchronize repository code, and enforce deterministic seeds across all random generators.

> [!NOTE]
> **Decoupled Edge Prototype Architecture (Proposal Section 5 & 7):**  
> The system operates as a distributed prototype consisting of:
> 1. **Central Federated Server Process (`flower_server.py`)**: Runs aggregation algorithms, coordinates communication rounds, and evaluates global holdout metrics.
> 2. **Edge Client Processes (`flower_client.py`)**: Independent OS processes training locally on private partitions, communicating strictly over **gRPC network sockets** (TCP/IP Port 8080).


In [ ]:
import os, sys, subprocess, warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Compute Device: {device} | PyTorch: {torch.__version__}')

# Colab setup
if os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    repo_dir = '/content/FedLearning'
    if not os.path.exists(repo_dir):
        subprocess.run(['git', 'clone', 'https://github.com/NVKQ2022/FedLearning.git', repo_dir], check=True)
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
elif os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

try:
    import flwr as fl
    print(f'✅ Flower Version: {fl.__version__}')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', 'flwr>=1.13.0'], check=True)
    import flwr as fl
    print(f'✅ Flower installed: {fl.__version__}')

from src.utils.seed import set_seed
set_seed(42)
print('✅ Deterministic random seed locked at 42.')

---
### ⚙️ Step 2: Cấu hình siêu tham số thí nghiệm (Mục 8 Đề cương)

Cấu hình các tham số thực nghiệm cho các kịch bản từ **E2** đến **E6** thông qua lớp `Experiment` container:
* **Kịch bản E2**: IID FL với FedAvg ($K=5$ clients, phân chia IID).
* **Kịch bản E3**: Non-IID Dirichlet $\alpha = 1.0$ với FedAvg.
* **Kịch bản E4**: Non-IID Dirichlet $\alpha = 0.5$ với FedAvg.
* **Kịch bản E5**: Non-IID Dirichlet $\alpha = 0.1$ với FedProx ($\mu = 0.05$).
* **Kịch bản E6**: Đánh giá mở rộng số lượng client $K \in \{5, 7, 10\}$.

> [!TIP]
> **Dễ dàng chuyển đổi kịch bản chỉ với `CONFIG`:**  
> Toàn bộ pipeline (phân chia dữ liệu, kiến trúc mô hình, xuất scenario và thực thi gRPC qua `run_flower_grpc(config=CONFIG)`) đều được điều khiển trực tiếp bởi đối tượng `CONFIG`. Bạn chỉ cần thay đổi cấu hình tại đây (hoặc dùng hàm tiện ích `Experiment.for_scenario("E5")`), toàn bộ các bước tiếp theo sẽ tự động điều chỉnh theo đúng kịch bản mong muốn.


In [ ]:
from src.utils.config import (
    ExperimentConfig,
    DataConfig,
    ModelConfig,
    LossConfig,
    OptimizerConfig,
    FederatedConfig,
    FedAvg,
    FedProx,
)

CONFIG = ExperimentConfig(
    experiment_name="flower_e2_fedavg_iid",
    seed=42,
    data=DataConfig(
        sample_size=100000,                 # 100k samples for fast execution
        test_size=0.3,                      # 30% upcoming / unseen data (Holdout Test Set at Server)
        val_size=0.1,                       # 10% centralized server validation set
        scaler_type="robust",               # 'robust' (median/IQR)
        batch_size=64,                      # Client local batch size
    ),
    model=ModelConfig(
        hidden_dims=(128, 64),              # Lightweight edge MLP (~15k params)
        dropout_rate=0.2,
    ),
    loss=LossConfig(
        loss_type="focal_loss",             # MultiClassFocalLoss
        class_weight_strategy="balanced",   # Inverse frequency class weighting
        focal_gamma=2.0,
    ),
    optimizer=OptimizerConfig(
        optimizer_type="adamw",
        learning_rate=1e-3,
        weight_decay=1e-4,
        max_grad_norm=5.0,
    ),
    federated=FederatedConfig(
        algorithm=FedAvg(),                 # Algorithm inside: FedAvg() or FedProx(mu=0.05)
        num_clients=5,                      # Common FL parameter: edge clients K
        num_rounds=10,                      # Common FL parameter: server rounds T
        local_epochs=2,                     # Common FL parameter: local epochs E
        partition_type="iid",               # 'iid' or 'dirichlet'
        dirichlet_alpha=0.5,                # Dirichlet concentration parameter (0.5 moderate, 0.1 severe)
        fraction_fit=1.0,                   # Train on 100% of available clients
        fraction_evaluate=1.0,              # 0.0: Local validation runs inside fit() (zero extra gRPC round-trip)
        client_val_ratio=0.1,               # Each client splits 70% train / 10% val / 20% test
        client_test_ratio=0.2,
    ),
)

CONFIG.display()


---
### 💾 Step 3: Tiền xử lý dữ liệu và Phân chia IID / Non-IID (Mục 2 & Mục 4 Đề cương)

> [!IMPORTANT]
> **Kiến trúc phân chia 70% Federated Partitions & 30% Upcoming Data:**  
> * **70% Dữ liệu (Federated Partitions):** Được phân chia cho $K$ client (theo IID hoặc Dirichlet Non-IID). Từng client tự chia thành **80% Train** (huấn luyện cục bộ) và **20% Val** (đánh giá **Tính cá nhân hóa - Personalization**).
> * **30% Dữ liệu (Upcoming / Unseen Data):** Giữ độc lập tại Server làm **Holdout Test Set** để đánh giá **Tính tổng quát (Generalization)** của mô hình toàn cục sau huấn luyện (mô phỏng luồng tấn công tương lai / zero-day chưa từng thấy).

#### 1. Quy trình tiền xử lý dữ liệu không rò rỉ (Mục 2.2):
* Loại bỏ dữ liệu trùng lặp và các giá trị không hợp lệ.
* Xử lý giá trị thiếu.
* Loại bỏ các thuộc tính không cần thiết hoặc có nguy cơ làm rò rỉ nhãn.
* Chuẩn hóa các đặc trưng số: Các tham số chuẩn hóa chỉ được xác định từ tập train, sau đó áp dụng thống nhất cho validation, test và dữ liệu của các client.
* Xử lý mất cân bằng lớp bằng trọng số phân lớp theo phân phối nhãn tập train.

#### 2. Phân chia dữ liệu IID và Non-IID (Mục 4.1 & 4.2):
* **Phân chia IID (Mục 4.1):** Dữ liệu được phân chia sao cho các client có phân phối nhãn gần tương đương nhau (Kịch bản E2).
* **Phân chia Non-IID bằng phân phối Dirichlet (Mục 4.2):** $p_c$ biểu diễn tỷ lệ mẫu của lớp $c$ được phân bổ cho các client:
  * $\alpha = 1.0$: Mức độ không đồng nhất thấp (Kịch bản E3).
  * $\alpha = 0.5$: Mức độ không đồng nhất trung bình (Kịch bản E4).
  * $\alpha = 0.1$: Mức độ không đồng nhất cao (Kịch bản E5).


In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights
from src.data.partitioner import (
    StratifiedIIDPartitioner,
    DirichletNonIIDPartitioner,
    create_client_dataloaders,
    summarize_client_partitions
)
from src.eda.visualizer import plot_partition_heterogeneity
import os
import pandas as pd

# Locate dataset CSV
candidate_paths = [
    CONFIG.gdrive_csv,
    CONFIG.alt_gdrive_csv,
    CONFIG.target_csv_path,
    'datasets/CICIOT2023/merged_CICIOT2023_data.csv',
    'datasets/merged_CICIOT2023_data.csv',
]
csv_path = next((p for p in candidate_paths if os.path.exists(p) and os.path.getsize(p) > 10*1024*1024), None)
if not csv_path:
    csv_path = CONFIG.target_csv_path

print(f"📁 Loading and preprocessing from: {csv_path}")
data = load_and_preprocess_ciciot2023(
    csv_path=csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_weights = compute_balanced_class_weights(data['y_train'], data['num_classes'], strategy=CONFIG.class_weight_strategy)
class_names = data['class_names']
minority_classes = ['Web-based', 'Brute-force']

# Generate client partitions
if CONFIG.partition_type == "iid":
    partitioner = StratifiedIIDPartitioner(num_clients=CONFIG.num_clients, seed=CONFIG.seed)
else:
    partitioner = DirichletNonIIDPartitioner(
        num_clients=CONFIG.num_clients,
        alpha=CONFIG.dirichlet_alpha,
        min_samples_per_client=100,
        seed=CONFIG.seed
    )

client_partitions = partitioner.partition(data['X_train'], data['y_train'])
client_loaders = create_client_dataloaders(data['X_train'], data['y_train'], client_partitions, batch_size=CONFIG.batch_size)

# Bảng phân phối và tính toán Entropy (Đề cương Mục 4)
summary_df = summarize_client_partitions(client_partitions, data['y_train'], class_names=class_names)

print("=" * 100)
print(f"📊 BẢNG PHÂN PHỐI DỮ LIỆU CÁC CLIENT (Partition: {CONFIG.partition_type.upper()})")
print("=" * 100)
# Hiển thị các cột thông tin chung và tỷ lệ phần trăm của từng class
pct_cols = [c for c in summary_df.columns if c.endswith('_pct') and c != 'dominant_pct']
display_cols = ['client_id', 'total_samples', 'dominant_class', 'entropy'] + pct_cols

pd.options.display.float_format = '{:,.2f}%'.format
display_df = summary_df[display_cols].copy()
display_df.columns = [c.replace('_pct', ' (%)') for c in display_df.columns]
display(display_df)
pd.reset_option('display.float_format')

# Trực quan hóa độ dị đồng nhất (Heterogeneity) bằng biểu đồ Stacked Bar
print("\n")
plot_partition_heterogeneity(
    summary_df=summary_df,
    class_names=class_names,
    title=f"Phân phối dị đồng nhất giữa các Clients (Non-IID $\\alpha={CONFIG.dirichlet_alpha}$)" if CONFIG.partition_type != 'iid' else "Phân phối đồng nhất giữa các Clients (IID)",
    show=True
)


---
### 🏛️ Step 4: Kiến trúc mô hình MLP cục bộ (Mục 3 Đề cương)

Một mô hình MLP tập trung được xây dựng trước để kiểm tra quy trình tiền xử lý và làm baseline so sánh với Federated Learning. Mô hình tập trung và mô hình tại các client sử dụng cùng kiến trúc.

#### Bảng 3: Kiến trúc MLP tại mỗi client (Trích Đề cương chi tiết KLTN)
| Thành phần | Thiết kế dự kiến |
| :--- | :--- |
| **Input** | Số lượng network-flow features sau preprocessing (39 đặc trưng) |
| **Hidden 1** | Linear(input_dim, 128) + ReLU |
| **Regularization** | Dropout |
| **Hidden 2** | Linear(128, 64) + ReLU |
| **Output** | Linear(64, number_of_classes) |
| **Loss** | Cross-Entropy hoặc weighted variant nếu cần xử lý imbalance |
| **Optimizer** | Adam / SGD; cố định trong các baseline |

Trong mỗi communication round, client nhận global model từ server, huấn luyện trên dữ liệu cục bộ và gửi các tham số mô hình đã cập nhật về server.


In [ ]:
from src.models.tabular_mlp import TabularIoTMLPModel
from src.losses.focal_loss import build_loss_function
from src.federated.flower_server import build_flower_server_eval_fn

global_model = TabularIoTMLPModel(
    input_dim=data['input_dim'],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data['num_classes'],
    dropout_rate=CONFIG.dropout_rate
).to(device)

criterion = build_loss_function(
    loss_type=CONFIG.loss_type,
    class_weights=class_weights if CONFIG.class_weight_strategy != 'none' else None,
    gamma=CONFIG.focal_gamma,
    device=device
)

model_size_bytes = sum(p.numel() * p.element_size() for p in global_model.parameters())

# Server-side holdout evaluation callback executed after every communication round
flower_eval_fn = build_flower_server_eval_fn(
    model=global_model,
    val_loader=data['val_loader'],
    class_names=class_names,
    device=device
)

print(f"✅ Global Model: {global_model.get_num_parameters():,} params ({model_size_bytes/1024:.2f} KB)")
print("✅ Server Holdout Evaluation Callback built successfully.")

---
### 📦 Step 5: Xuất cấu trúc Kịch bản & Phân chia Partitions cho gRPC

Trước khi khởi chạy các tiến trình gRPC độc lập, toàn bộ cấu trúc kịch bản được xuất ra thư mục `scenarios/<scenario_name>/`.
Điều này đảm bảo tính cách ly dữ liệu vật lý (data isolation) tuyệt đối cho từng client tiến trình $k \in \{0, \dots, K-1\}$ và Server.

> [!NOTE]
> **Quy hoạch phân vùng dữ liệu khoa học:**
> * **Client Partitions (70% Dữ liệu):** Mỗi client nhận một partition riêng biệt và tự chia thành **80% Train** (`partition.npz`) để huấn luyện và **20% Val** (`val_partition.npz`) để kiểm thử tính cá nhân hóa (Personalization).
> * **Server Central Holdout (Validation Split):** Dùng để Server đánh giá trung tâm sau mỗi round và checkpoint `best_weights.npz` (`val_data.npz`).
> * **Server Global Test Set (30% Upcoming Data):** Được lưu độc lập tại Server (`test_data.npz` / `global_test.npz`), đại diện cho dữ liệu luồng mạng tương lai hoàn toàn cô lập với các client để kiểm thử mô hình tốt nhất (Best Global Model) ở Step 8.


In [ ]:
from src.federated.scenario import create_federated_scenario
import os

scenario_dir = f"scenarios/{CONFIG.experiment_name}"
print(f"📦 Generating and exporting scenario structure for gRPC: {scenario_dir}")

# Exports scenario partitions and config using FederatedConfig directly (with 80/20 local train/val split & 30% upcoming test set)
create_federated_scenario(
    scenario_name=CONFIG.experiment_name,
    client_partitions=client_partitions,
    X_train=data["X_train"],
    y_train=data["y_train"],
    X_val=data["X_val"],
    y_val=data["y_val"],
    X_test=data["X_test"],
    y_test=data["y_test"],
    class_names=class_names,
    config=CONFIG.federated,
    base_dir="scenarios",
    generate_plots=True,
    client_val_ratio=CONFIG.federated.client_val_ratio,
    client_test_ratio=CONFIG.federated.client_test_ratio
)

print(f"✅ Scenario artifacts ready at: {scenario_dir}/")
print(f"   - Server Central Holdout Val:    {scenario_dir}/server/val_data.npz")
print(f"   - Server Global Test (30% Unseen): {scenario_dir}/server/test_data.npz")
print(f"   - Client Train Partitions (70%): {scenario_dir}/client_*/partition.npz")
print(f"   - Client Val Partitions (10%):   {scenario_dir}/client_*/val_partition.npz")
print(f"   - Client Test Partitions (20%):  {scenario_dir}/client_*/test_partition.npz")


---
### 🌐 Step 6: Multi-Process Flower gRPC Execution & Phương pháp tổng hợp (Mục 6 Đề cương)

Server Flower (`0.0.0.0:8080`) và $K$ Client chạy như các tiến trình OS độc lập giao tiếp qua **gRPC network sockets** (TCP/IP).

#### 1. FedAvg (Mục 6.1 Đề cương):
FedAvg tổng hợp các mô hình cục bộ theo số lượng mẫu của từng client:

$$\mathbf{w}_{t+1} = \sum_{k=1}^K \frac{n_k}{N} \mathbf{w}_{t+1}^{(k)}$$

*Giải thích các biến theo Mục 6.1 Đề cương:*
* $n_k$: là số lượng mẫu của client $k$. Client có nhiều dữ liệu hơn sẽ đóng góp trọng số lớn hơn vào global model.
* $\mathbf{w}_{t+1}$: là trọng số tiếp theo của model sau khi tổng hợp từ các client.
* $\mathbf{w}_{t+1}^{(k)}$: là trọng số của model của client thứ $k$.
* $N$: là tổng số mẫu dữ liệu của tất cả các client ($N = \sum_{k=1}^K n_k$).
* $K$: là tổng số client tham gia vào quá trình huấn luyện.

#### 2. FedProx (Mục 6.2 Đề cương):
FedProx bổ sung một thành phần proximal vào hàm mất mát cục bộ:

$$\min_{\mathbf{w}} F_k(\mathbf{w}) + \frac{\mu}{2} \|\mathbf{w} - \mathbf{w}_t\|^2$$

*Giải thích các biến theo Mục 6.2 Đề cương:*
* $\mu$: là hệ số proximal.
* $\mathbf{w}_t$: là global model tại đầu round.
* $F_k(\mathbf{w})$: hàm mất mát cục bộ của client $k$.
* Thành phần $\frac{\mu}{2} \|\mathbf{w} - \mathbf{w}_t\|^2$: hạn chế local model thay đổi quá xa global model, từ đó hỗ trợ huấn luyện trong điều kiện dữ liệu không đồng nhất.

#### 3. Đánh giá tính cá nhân hóa (Personalization) tối ưu không tốn overhead:
Thay vì mở thêm một phase gRPC riêng biệt (`fraction_evaluate = 0.0`), ngay sau khi hoàn thành huấn luyện cục bộ trong `fit()`, mỗi client tự inference trên **tập kiểm thử cục bộ 20%** ($D_{\text{val}}^{(k)}$) của riêng mình:

$$\text{Acc}_k^{(t+1)} = \frac{1}{|D_{\text{val}}^{(k)}|} \sum_{(\mathbf{x}, y) \in D_{\text{val}}^{(k)}} \mathbb{I}\left(\arg\max f(\mathbf{x}; \mathbf{w}_{t+1}^{(k)}) = y\right)$$

Client đính kèm `val_accuracy` vào gói tin gửi về Server. Server tổng hợp tham số và in ngay bảng so sánh tính cá nhân hóa của các client sau mỗi round mà không tốn thêm bất kỳ round giao tiếp mạng nào!


In [ ]:
import time
from src.federated.grpc_runner import run_flower_grpc

sim_start = time.perf_counter()

# Executes Flower distributed gRPC training using FederatedConfig from CONFIG.federated
round_history = run_flower_grpc(
    scenario_name=CONFIG.experiment_name,
    federated_config=CONFIG.federated,
    device=device.type if hasattr(device, "type") else str(device),
    stream_logs=True,
    auto_find_port=True
)

sim_time = time.perf_counter() - sim_start
print("=" * 80)
print(f"🎉 Flower gRPC Training completed in {sim_time:.1f}s ({sim_time/60:.2f} min)!")
if round_history and "val_macro_f1" in round_history and len(round_history["val_macro_f1"]) > 0:
    best_f1 = max(round_history["val_macro_f1"])
    best_rnd = round_history["val_macro_f1"].index(best_f1) + 1
    print(f"Best Round: {best_rnd} | Best Val Macro-F1: {best_f1 * 100:.2f}%")


---
### 📈 Step 7: Theo dõi tiến trình hội tụ và Chi phí truyền thông (Mục 9 Đề cương)

Đánh giá các chỉ số hệ thống theo **Bảng 9: Các chỉ số đánh giá hệ thống** (Convergence rounds, Training time, Communication cost, CPU/RAM/GPU memory, Scalability).

#### Hình (Mục 9): Công thức tính chi phí truyền thông (Trích Đề cương chi tiết KLTN)
$$C_{\text{total}} = \sum_{t=1}^T 2 m_t B$$

*Giải thích các biến theo Mục 9 Đề cương:*
* $m_t$: là số client tham gia round $t$.
* $B$: là kích thước mô hình.
* $T$: là tổng số communication round.
* Hệ số $2$: phản ánh chi phí truyền thông 2 chiều (tải xuống mô hình toàn cục từ server và tải lên cập nhật mô hình từ client trong mỗi round).


In [ ]:
import matplotlib.pyplot as plt

rounds = round_history.get("round", [])

if len(rounds) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=120)
    
    # Left: Macro-F1 Convergence
    val_f1 = round_history["val_macro_f1"][-len(rounds):]
    ax1.plot(rounds, [f * 100 for f in val_f1], 'o-', color='#2980b9', linewidth=2)
    ax1.set_title(f"Flower {CONFIG.federated_strategy.upper()} Macro-F1 Convergence (gRPC)", fontsize=11, fontweight='bold')
    ax1.set_xlabel("Round", fontsize=10)
    ax1.set_ylabel("Macro-F1 (%)", fontsize=10)
    ax1.grid(True, linestyle="--", alpha=0.5)
    
    # Right: Cumulative Communication Overhead (MB)
    comm_mb = round_history.get("comm_cost_mb", [0.0] * len(rounds))
    ax2.plot(rounds, comm_mb, 's-', color='#27ae60', linewidth=2)
    ax2.set_title("Cumulative Network Communication Overhead (MB)", fontsize=11, fontweight='bold')
    ax2.set_xlabel("Round", fontsize=10)
    ax2.set_ylabel("Transferred (MB)", fontsize=10)
    ax2.grid(True, linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    print(f"📊 Total Payload Exchanged: {comm_mb[-1]:.2f} MB")

#### 📊 Bảng và biểu đồ so sánh độ chính xác của từng client qua các rounds (Decentralized Evaluation)
Theo dõi tiến trình độ chính xác cục bộ của từng client khi đánh giá mô hình toàn cục trên tập kiểm thử riêng (20% partition):


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

client_eval = round_history.get("client_eval", {})
client_accuracies = round_history.get("client_accuracies", {})
rounds = round_history.get("round", [])

if client_eval and len(rounds) > 0:
    # 1. Bảng so sánh độ chính xác của từng client qua các rounds
    table_data = []
    for r in rounds:
        r_str = str(r)
        row = {"Round": r}
        if r_str in client_eval:
            r_metrics = client_eval[r_str]
            accs = []
            for cid in sorted(r_metrics.keys(), key=lambda x: int(x) if str(x).isdigit() else str(x)):
                acc_val = r_metrics[cid]["accuracy"] * 100
                row[f"Client {cid} Acc (%)"] = round(acc_val, 2)
                accs.append(acc_val)
            if accs:
                row["Mean Client Acc (%)"] = round(sum(accs) / len(accs), 2)
                row["Spread (Max-Min) (%)"] = round(max(accs) - min(accs), 2)
        table_data.append(row)
    
    df_clients = pd.DataFrame(table_data)
    print("=" * 88)
    print("📋 BẢNG SO SÁNH ĐỘ CHÍNH XÁC TỪNG CLIENT QUA CÁC ROUNDS (DECENTRALIZED EVALUATION)")
    print("=" * 88)
    display(df_clients)
    
    # 2. Biểu đồ so sánh đường cong hội tụ của từng Client vs Central Holdout Validation
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.8), dpi=120)
    
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2"]
    markers = ["o", "s", "^", "D", "v", "<", ">"]
    
    for i, (ckey, acc_list) in enumerate(sorted(client_accuracies.items())):
        c_label = ckey.replace("_", " ").title()
        y_vals = [a * 100 for a in acc_list[-len(rounds):]]
        ax1.plot(rounds, y_vals, marker=markers[i % len(markers)], color=colors[i % len(colors)],
                 linewidth=1.8, label=c_label)
    
    # Đường đánh giá tập validation toàn cục (Server central holdout)
    if "val_acc" in round_history and len(round_history["val_acc"]) >= len(rounds):
        central_acc = [a * 100 for a in round_history["val_acc"][-len(rounds):]]
        ax1.plot(rounds, central_acc, "k--", linewidth=2.5, label="Global Central Val")
        
    ax1.set_title("Tiến trình độ chính xác của từng Client qua các Rounds", fontsize=11, fontweight="bold")
    ax1.set_xlabel("Communication Round", fontsize=10)
    ax1.set_ylabel("Validation Accuracy (%)", fontsize=10)
    ax1.legend(loc="lower right", framealpha=0.9)
    ax1.grid(True, linestyle="--", alpha=0.5)
    
    # Phân bố độ chính xác của các client tại round cuối
    last_r_str = str(rounds[-1])
    if last_r_str in client_eval:
        last_metrics = client_eval[last_r_str]
        sorted_cids = sorted(last_metrics.keys(), key=lambda x: int(x) if str(x).isdigit() else str(x))
        client_names = [f"Client {cid}" for cid in sorted_cids]
        final_accs = [last_metrics[cid]["accuracy"] * 100 for cid in sorted_cids]
        
        bars = ax2.bar(client_names, final_accs, color=colors[:len(client_names)], alpha=0.85, edgecolor="black")
        ax2.set_title(f"Độ chính xác các Client tại Round {rounds[-1]} (Non-IID Skew)", fontsize=11, fontweight="bold")
        ax2.set_ylabel("Accuracy (%)", fontsize=10)
        ax2.set_ylim(0, 105)
        ax2.grid(axis="y", linestyle="--", alpha=0.5)
        
        for bar in bars:
            h = bar.get_height()
            ax2.annotate(f"{h:.1f}%",
                         xy=(bar.get_x() + bar.get_width() / 2, h),
                         xytext=(0, 3), textcoords="offset points",
                         ha="center", va="bottom", fontsize=9, fontweight="bold")
    
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Decentralized evaluation history không có sẵn.")


---
### 🏆 Step 8: Đánh giá mô hình tốt nhất (Best Global Model) trên Global Test Set (30% Upcoming Data) (Mục 9 Đề cương)

Tại bước này, chúng ta sử dụng **Global Test Set (30% Upcoming / Unseen Data)** được lưu độc lập tại Server để đánh giá toàn diện **Mô hình toàn cục tốt nhất (Best Global Model)** thu được từ quá trình huấn luyện Federated Learning:

> [!IMPORTANT]
> **Quy trình đánh giá khoa học & bảo đảm tính khách quan:**
> 1. **Khôi phục Best Global Checkpoint (`best_weights.npz`):** Server tự động lưu lại bộ trọng số mô hình toàn cục tại round đạt Macro-F1 cao nhất trên tập validation nội bộ (`scenarios/<scenario_name>/server/best_weights.npz`).
> 2. **Global Test Set độc lập (30% Upcoming Data):** Đại diện cho dữ liệu mạng tương lai chưa từng xuất hiện trong quá trình huấn luyện tại bất kỳ client nào (`scenarios/<scenario_name>/server/test_data.npz`).
> 3. **Đánh giá tính tổng quát hóa (Generalization) & Phát hiện tấn công thiểu số:** Kiểm tra xem mô hình có bị thiên lệch (bias) về các lớp phổ biến hay vẫn giữ được khả năng nhận diện các cuộc tấn công tinh vi (như `Web-based`, `Brute-force`).

#### Bảng 9: Các chỉ số đánh giá mô hình (Trích Đề cương chi tiết KLTN)
| Chỉ số (Metric) | Công thức / Bản chất | Ý nghĩa thực tiễn theo Đề cương |
| :--- | :--- | :--- |
| **Accuracy** | $\frac{TP + TN}{Total}$ | Hiệu năng tổng thể của mô hình. |
| **Macro-Precision** | $\frac{1}{C}\sum_{c=1}^C \text{Precision}_c$ | Độ chính xác trung bình giữa các lớp không bị chi phối bởi kích thước mẫu. |
| **Macro-Recall** | $\frac{1}{C}\sum_{c=1}^C \text{Recall}_c$ | Tỷ lệ phát hiện trung bình trên toàn bộ các lớp tấn công. |
| **Macro-F1** | $\frac{1}{C}\sum_{c=1}^C \text{F1}_c$ | Chỉ số cốt lõi đánh giá dữ liệu mất cân bằng nghiêm trọng trong IoT. |
| **Weighted-F1** | $\sum_{c=1}^C \frac{n_c}{N} \text{F1}_c$ | F1 có tính đến tỷ trọng thực tế của từng lớp. |
| **Minority Recall** | $\text{Recall}_{\text{rare}}$ | Khả năng phát hiện các luồng xâm nhập hiếm gặp (`Web-based`, `Brute-force`). |
| **Confusion Matrix** | Ma trận $C \times C$ | Phân tích chi tiết nhầm lẫn giữa các attack classes. |


In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from torch.utils.data import TensorDataset, DataLoader

from src.evaluation.evaluator import evaluate_comprehensive
from src.visualization import plot_confusion_matrix
from src.federated.scenario import load_server_test_data

scenario_dir = f"scenarios/{CONFIG.experiment_name}"

# 1. Khôi phục trọng số mô hình tốt nhất (Best Global Model Checkpoint)
best_ckpt_candidates = [
    f"{scenario_dir}/server/best_weights.npz",
    f"checkpoints/flower_{CONFIG.federated_strategy}_best_weights.npz",
    f"checkpoints/best_weights.npz"
]
best_ckpt = next((p for p in best_ckpt_candidates if os.path.exists(p)), None)

if best_ckpt:
    npz = np.load(best_ckpt)
    best_weights = [npz[f"arr_{i}"] for i in range(len(npz.files))]
    global_model.set_weights(best_weights)
    print(f"✅ Đã khôi phục thành công Best Global Model Checkpoint từ: {best_ckpt}")
else:
    print("ℹ️ Chưa tìm thấy file checkpoint riêng, sử dụng trọng số Global Model từ round huấn luyện cuối.")

# 2. Chuẩn bị Global Test DataLoader (từ in-memory data['test_loader'] hoặc đọc từ test_data.npz)
if 'test_loader' in data and data['test_loader'] is not None:
    test_loader = data['test_loader']
    num_test_samples = len(data['y_test']) if 'y_test' in data else 'N/A'
    print(f"📦 Sử dụng in-memory Global Test DataLoader ({num_test_samples} mẫu).")
else:
    X_test_arr, y_test_arr, meta_test = load_server_test_data(scenario_dir)
    if X_test_arr is not None:
        test_ds = TensorDataset(torch.from_numpy(X_test_arr), torch.from_numpy(y_test_arr))
        test_loader = DataLoader(test_ds, batch_size=CONFIG.batch_size, shuffle=False)
        print(f"📦 Đã tải Global Test Set từ {scenario_dir}/server/test_data.npz ({len(y_test_arr):,} mẫu).")
    else:
        raise FileNotFoundError("Không tìm thấy Global Test Set! Vui lòng kiểm tra lại cấu hình dữ liệu.")

# 3. Thực hiện đánh giá toàn diện trên Global Test Set (30% Upcoming Data)
test_results = evaluate_comprehensive(
    model=global_model,
    dataloader=test_loader,
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

# 4. Hiển thị bảng tổng kết kết quả đánh giá (Đề cương Mục 9)
summary_metrics = {
    "Chỉ số đánh giá (Metric)": [
        "Test Accuracy (%)",
        "Macro-F1 Score (%)",
        "Weighted-F1 Score (%)",
        "Macro-Precision (%)",
        "Macro-Recall (%)"
    ],
    "Kết quả (%)": [
        round(test_results['accuracy'] * 100, 2),
        round(test_results['macro_f1'] * 100, 2),
        round(test_results['weighted_f1'] * 100, 2),
        round(test_results['macro_precision'] * 100, 2),
        round(test_results['macro_recall'] * 100, 2)
    ],
    "Ý nghĩa theo Đề cương KLTN": [
        "Hiệu năng tổng thể trên 30% dữ liệu tương lai",
        "Chỉ số cân bằng cốt lõi (không thiên lệch lớp đa số)",
        "F1 có trọng số theo tỷ lệ phân bố thực tế",
        "Độ chuẩn xác trung bình giữa các lớp tấn công",
        "Tỷ lệ phát hiện trung bình trên toàn bộ các lớp"
    ]
}
df_summary = pd.DataFrame(summary_metrics)

print("=" * 82)
print(f"🏆 KẾT QUẢ ĐÁNH GIÁ BEST GLOBAL MODEL TRÊN GLOBAL TEST SET ({CONFIG.federated_strategy.upper()})")
print("=" * 82)
display(df_summary)

print("\n🎯 Khả năng phát hiện các lớp tấn công thiểu số (Minority Attack Classes):")
for cls_name, rec_val in test_results['minority_recall'].items():
    print(f"   • {cls_name:<16}: {rec_val * 100:6.2f}%")
print("=" * 82)

# 5. Lưu kết quả kiểm thử vào server/test_results.json
test_results_export = {k: v for k, v in test_results.items() if k != 'confusion_matrix'}
test_results_path = os.path.join(scenario_dir, "server", "test_results.json")
with open(test_results_path, "w") as f:
    json.dump(test_results_export, f, indent=2)
print(f"💾 Đã lưu kết quả kiểm thử chi tiết vào: {test_results_path}")

# 6. Trực quan hóa Confusion Matrix & Per-Class Recall Bar Chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.2), dpi=120)

# Confusion Matrix Heatmap
cm = np.array(test_results['confusion_matrix'])
cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-12)
im = ax1.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
ax1.set_title(f"Confusion Matrix: Best Global Model ({CONFIG.federated_strategy.upper()})", fontsize=11, fontweight="bold")
plt.colorbar(im, ax=ax1, fraction=0.046, pad=0.04)
tick_marks = np.arange(len(class_names))
ax1.set_xticks(tick_marks)
ax1.set_xticklabels(class_names, rotation=35, ha='right', fontsize=9)
ax1.set_yticks(tick_marks)
ax1.set_yticklabels(class_names, fontsize=9)
ax1.set_ylabel('True Label', fontsize=10)
ax1.set_xlabel('Predicted Label', fontsize=10)

for i in range(len(class_names)):
    for j in range(len(class_names)):
        color = "white" if cm_norm[i, j] > 0.5 else "black"
        ax1.text(j, i, f"{cm[i, j]:,}\n({cm_norm[i, j]*100:.1f}%)",
                 ha="center", va="center", color=color, fontsize=8)

# Per-Class Recall Bar Chart (nhấn mạnh minority attacks)
class_recalls = cm.diagonal() / (cm.sum(axis=1) + 1e-12) * 100
bar_colors = ['#d62728' if name in minority_classes else '#1f77b4' for name in class_names]
bars = ax2.bar(class_names, class_recalls, color=bar_colors, alpha=0.85, edgecolor='black')
ax2.set_title("Tỷ lệ nhận diện (Recall) theo từng lớp tấn công", fontsize=11, fontweight="bold")
ax2.set_ylabel("Recall (%)", fontsize=10)
ax2.set_ylim(0, 108)
ax2.set_xticklabels(class_names, rotation=35, ha='right', fontsize=9)
ax2.grid(axis='y', linestyle='--', alpha=0.5)

for bar in bars:
    h = bar.get_height()
    ax2.annotate(f"{h:.1f}%",
                 xy=(bar.get_x() + bar.get_width() / 2, h),
                 xytext=(0, 3), textcoords="offset points",
                 ha="center", va="bottom", fontsize=8.5, fontweight="bold")

red_patch = mpatches.Patch(color='#d62728', label='Minority Attack (Web-based, Brute-force)')
blue_patch = mpatches.Patch(color='#1f77b4', label='Majority / Standard Class')
ax2.legend(handles=[red_patch, blue_patch], loc='lower right', framealpha=0.9)

plt.tight_layout()
plt.show()


---
### 💾 Step 9: Export Scenario Results & Scientific Artifacts
Persists evaluation metrics, convergence history, and holdout performance matrices into `scenarios/<scenario_name>/server/metrics.json` for thesis reporting and cross-scenario ablation comparisons.


In [ ]:
import json

scenario_dir = f"scenarios/{CONFIG.experiment_name}"
metrics_export = {k: v for k, v in test_results.items() if k != 'confusion_matrix'}
metrics_export['round_history'] = round_history
metrics_path = f"{scenario_dir}/server/metrics.json"

with open(metrics_path, 'w') as f:
    json.dump(metrics_export, f, indent=2)

print(f"✅ Successfully updated scenario metrics: {metrics_path}")
print(f"\n🎉 Flower {CONFIG.federated_strategy.upper()} gRPC Pipeline complete!")
